### Create a tf-idf-based classificator model for the bank77 dataset

In [10]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import GradientBoostingClassifier
from utils.utils import preprocessing

In [11]:
import os
import json
import pandas as pd

def extract_input_texts_from_folder(folder_path):
    records = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".jsonl"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        record = json.loads(line)
                        custom_id = record.get("custom_id")
                        messages = record.get("body", {}).get("messages", [])
                        for message in messages:
                            if message.get("role") == "user":
                                user_content = message.get("content", "")
                                nested_json = json.loads(user_content)
                                input_text = nested_json.get("input_text")
                                if input_text:
                                    records.append({
                                        "custom_id": custom_id,
                                        "input_text": input_text
                                    })
                    except Exception as e:
                        print(f"Error in file {filename}, skipping line: {e}")

    return pd.DataFrame(records)


In [12]:
from datasets import load_dataset

data_load = load_dataset("banking77")

In [13]:
df_train = pd.DataFrame(data_load['train'])
df_test = pd.DataFrame(data_load['test'])

In [14]:
df_test['text'] = df_test.text.apply(lambda x: preprocessing(x))
df_train['text'] = df_train.text.apply(lambda x: preprocessing(x))

In [15]:
# # Create a tf-idf matrix
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_train['text'])
y = df_train['label']

In [16]:
# # Train a classifier
clf = GradientBoostingClassifier(random_state=42)
clf.fit(X, y)

GradientBoostingClassifier(random_state=42)

In [17]:
# Test the classifier
X_test = vectorizer.transform(df_test['text'])
y_test = df_test['label']
y_pred = clf.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.95      0.96        40
           1       0.91      0.97      0.94        40
           2       0.97      0.97      0.97        40
           3       0.73      0.60      0.66        40
           4       0.94      0.82      0.88        40
           5       0.45      0.78      0.57        40
           6       0.86      0.95      0.90        40
           7       0.83      0.85      0.84        40
           8       0.82      0.82      0.82        40
           9       1.00      0.93      0.96        40
          10       0.72      0.65      0.68        40
          11       0.54      0.80      0.65        40
          12       0.67      0.72      0.70        40
          13       0.92      0.90      0.91        40
          14       0.71      0.72      0.72        40
          15       0.65      0.70      0.67        40
          16       0.63      0.72      0.67        40
          17       0.81    